In [0]:
# =====================================================
# PARAMÉTRAGE - Widgets pour exécution via Databricks Job
# =====================================================

dbutils.widgets.text("catalog_name", "banking_lakehouse", "Catalog Unity Catalog")
dbutils.widgets.text("environment", "dev", "Environnement (dev/staging/prod)")

CATALOG = dbutils.widgets.get("catalog_name")
ENVIRONMENT = dbutils.widgets.get("environment")

print(f"✅ Paramètres reçus : catalog={CATALOG}, environment={ENVIRONMENT}")

In [0]:
# =====================================================
# Notebook : 07_build_fact_transactions
# Objectif : Construire la table de faits Transactions
#            (Domain 2 - Fraud Detection, anonymisé)
# Grain    : Une ligne = une transaction unique
# =====================================================

from pyspark.sql.functions import (
    col, when, lit, to_date, date_add, expr,
    hour, floor, round as spark_round
)

TABLE_TRANSACTIONS_SILVER = f"{CATALOG}.silver.transactions"
TABLE_DIM_DATE = f"{CATALOG}.gold.dim_date"
TABLE_FACT_TRANSACTIONS = f"{CATALOG}.gold.fact_transactions"

print("✅ Configuration fact_transactions chargée")
print(f"Source Silver : {TABLE_TRANSACTIONS_SILVER}")
print(f"Dimension Date: {TABLE_DIM_DATE}")
print(f"Cible Gold    : {TABLE_FACT_TRANSACTIONS}")

In [0]:
# =====================================================
# Enrichissement fact_transactions
# - Conversion Time (secondes) -> date_sk exploitable
# - Création de tranches de montant (analyse fraude)
# - Création de tranches horaires (heure de la journée)
# =====================================================

df_silver_tx = spark.table(TABLE_TRANSACTIONS_SILVER)

REFERENCE_DATE = "2026-01-01"

df_enriched = (
    df_silver_tx
    # --- Conversion Time (secondes écoulées) -> date calendaire ---
    .withColumn("days_elapsed", floor(col("Time") / 86400).cast("int"))  # 🆕 cast explicite en int
    .withColumn("transaction_date", expr(f"date_add(to_date('{REFERENCE_DATE}'), days_elapsed)"))
    .withColumn("date_sk", expr("cast(date_format(transaction_date, 'yyyyMMdd') as int)"))
    
    # --- Heure de la journée (0-23) ---
    .withColumn("seconds_in_day", col("Time") % 86400)
    .withColumn("hour_of_day", floor(col("seconds_in_day") / 3600).cast("int"))
    
    # --- Tranches de montant ---
    .withColumn(
        "amount_range",
        when(col("Amount") == 0, "Zero")
        .when(col("Amount") <= 50, "0-50")
        .when(col("Amount") <= 200, "50-200")
        .when(col("Amount") <= 1000, "200-1000")
        .otherwise("1000+")
    )
    
    # --- Libellé lisible pour la classe ---
    .withColumn(
        "transaction_type",
        when(col("Class") == 1, "Fraud").otherwise("Normal")
    )
)

print("📊 Aperçu de l'enrichissement :")
df_enriched.select(
    "transaction_id", "Time", "transaction_date", "date_sk",
    "hour_of_day", "Amount", "amount_range", "transaction_type"
).show(10, truncate=False)

print(f"\n📊 Nombre de lignes enrichies : {df_enriched.count()}")

In [0]:
# =====================================================
# Jointure avec dim_date + écriture finale fact_transactions
# Version AVEC IDEMPOTENCE (correction du bug x15)
# =====================================================

df_dim_date = spark.table(TABLE_DIM_DATE)

df_fact_final = (
    df_enriched.alias("fact")
    .join(
        df_dim_date.select("date_sk", "year", "quarter", "month_name", "day_name", "is_weekend").alias("dt"),
        col("fact.date_sk") == col("dt.date_sk"),
        "left"
    )
    .select(
        "fact.transaction_id",
        "fact.date_sk",
        "fact.transaction_date",
        "dt.year",
        "dt.quarter",
        "dt.month_name",
        "dt.day_name",
        "dt.is_weekend",
        "fact.hour_of_day",
        "fact.Time",
        "fact.Amount",
        "fact.amount_range",
        "fact.Class",
        "fact.transaction_type",
        "fact._silver_processed_at"
    )
)

# --- Vérification qualité de la jointure ---
nb_orphans = df_fact_final.filter(col("year").isNull()).count()
print(f"🔎 Lignes sans correspondance dans dim_date : {nb_orphans}")

# --- Écriture Gold AVEC IDEMPOTENCE ---
table_exists = spark.catalog.tableExists(TABLE_FACT_TRANSACTIONS)

if not table_exists:
    df_fact_final.write.format("delta").saveAsTable(TABLE_FACT_TRANSACTIONS)
    print(f"🆕 Table {TABLE_FACT_TRANSACTIONS} créée")
else:
    print(f"🔍 Table {TABLE_FACT_TRANSACTIONS} existe -> vérification idempotence")
    
    # 🆕 ANTI-JOIN : ne garde que les transaction_id absents de la table cible
    df_existing_ids_fact = spark.table(TABLE_FACT_TRANSACTIONS).select("transaction_id")
    
    df_new_only_fact = df_fact_final.join(
        df_existing_ids_fact,
        on="transaction_id",
        how="left_anti"
    )
    
    nb_new_fact = df_new_only_fact.count()
    nb_skipped_fact = df_fact_final.count() - nb_new_fact
    
    print(f"📊 Nouvelles transactions à ajouter : {nb_new_fact}")
    print(f"⏭️  Transactions déjà présentes (ignorées) : {nb_skipped_fact}")
    
    if nb_new_fact > 0:
        df_new_only_fact.write.format("delta").mode("append").saveAsTable(TABLE_FACT_TRANSACTIONS)
        print(f"✅ {nb_new_fact} nouvelle(s) transaction(s) ajoutée(s)")
    else:
        print("ℹ️ Aucune nouvelle transaction à ajouter (pipeline idempotent confirmé)")

nb_final = spark.table(TABLE_FACT_TRANSACTIONS).count()
print(f"\n✅ fact_transactions : {nb_final} lignes")